<a href="https://colab.research.google.com/github/aniray2908/satellite-esg-risk-engine/blob/main/experiments/python/ceri/ceri_v4_deployment_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CERI v4 — Deployment Tier Logic

This notebook converts the validated CERI v2 composite score
into a deterministic, deployable risk classification system.

Unlike clustering-based segmentation,
tier assignment here is threshold-based and stable.

Objective:
Translate composite risk score into operational tiers
with margin-based confidence.

In [1]:
import pandas as pd
import numpy as np

In [2]:
feature_df = pd.read_csv(
    "/content/drive/MyDrive/ceri_v2_feature_layer.csv"
)

In [3]:
def assign_tier(score):
    if score >= 0.5:
        return "High Risk"
    elif score <= -0.5:
        return "Low Risk"
    else:
        return "Moderate Risk"

feature_df["risk_tier"] = feature_df["CERI_z"].apply(assign_tier)

In [4]:
def confidence_margin(score):
    if score >= 0.5:
        return score - 0.5
    elif score <= -0.5:
        return -0.5 - score
    else:
        return min(score + 0.5, 0.5 - score)

feature_df["confidence_margin"] = feature_df["CERI_z"].apply(confidence_margin)

In [5]:
deployment_table = feature_df[[
    "asset",
    "CERI_z",
    "risk_tier",
    "confidence_margin"
]].sort_values("CERI_z", ascending=False)

deployment_table

,asset,CERI_z,risk_tier,confidence_margin
3,Grasberg,0.650290,High Risk,0.150290
0,Bingham,0.545452,High Risk,0.045452
1,Carajás,-0.112568,Moderate Risk,0.387432
2,Gevra,-1.083173,Low Risk,0.583173


In [6]:
def confidence_band(margin):
    if margin >= 0.3:
        return "High Confidence"
    elif margin >= 0.1:
        return "Medium Confidence"
    else:
        return "Low Confidence"

feature_df["confidence_band"] = feature_df["confidence_margin"].apply(confidence_band)

feature_df[[
    "asset",
    "CERI_z",
    "risk_tier",
    "confidence_margin",
    "confidence_band"
]].sort_values("CERI_z", ascending=False)

,asset,CERI_z,risk_tier,confidence_margin,confidence_band
3,Grasberg,0.650290,High Risk,0.150290,Medium Confidence
0,Bingham,0.545452,High Risk,0.045452,Low Confidence
1,Carajás,-0.112568,Moderate Risk,0.387432,High Confidence
2,Gevra,-1.083173,Low Risk,0.583173,High Confidence


## Phase 7 Summary — CERI v4 Deployment Layer

This notebook operationalizes the validated CERI v2 composite score into a deterministic risk classification system.

### What Was Achieved

- Converted composite exposure score (CERI_z) into stable risk tiers  
- Implemented threshold-based classification (no clustering dependency)  
- Introduced margin-based confidence scoring  
- Added confidence bands for interpretability  
- Produced a deployment-ready scoring table  

---

### Why Threshold-Based Tiering?

Clustering is valuable for validation and structural analysis, but deployment requires:

- Deterministic logic  
- Reproducibility without re-fitting models  
- Clear business interpretation  
- Stable decision boundaries  

Threshold-based tiering ensures scoring remains consistent across re-computations and portfolio updates.

---

### Governance Alignment

- CERI v2 remains the official scoring baseline  
- CERI v3 demonstrated robustness under weight optimization  
- CERI v4 translates the baseline score into an operational classification layer  

This preserves:

- Interpretability  
- Stability  
- Architectural discipline  
- Version separation  

---

### System Architecture Status

The exposure scoring framework now consists of:

1. Satellite Extraction Layer (GEE scripts)  
2. Feature Engineering Layer (F1, F2, F3)  
3. Composite Scoring Layer (CERI v2 baseline)  
4. Optimization & Robustness Layer (CERI v3)  
5. Deployment Tier Logic Layer (CERI v4)  

This represents a full modeling lifecycle from raw satellite imagery to deployable risk classification.

---

### Closing Note

The system demonstrates:

- Structural robustness  
- Controlled sensitivity under optimization  
- Geometric interpretability  
- Deterministic deployment logic  

Future phases may expand signal space or automate ingestion,
but the current framework provides a stable, governed foundation for scalable exposure-based risk scoring.